# Joint space Trajectory PID Control
- Get joint trajectory with sesquence of IK
- Follow with PID controller

#### 0. Generate Scene

In [1]:
import os
import sys
import numpy as np
import time
import mujoco
sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *
from pp_base_mujoco.SPEC_HELPER import *
from pp_base_mujoco.KINEMATICS import *

In [2]:
spec_helper = MjSpecHelper()
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, 0.7, 0),
    # r=(0, 0, -1.57),
    r=(0, 0, 0),
    prefix="",
    suffix="_right"
)
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, -0.7, 0),
    # r=(0, 0, 1.57),
    r=(0, 0, 0),
    prefix="",
    suffix="_left"
)
spec_helper.add_geom(
    name="box",
    type='box',
    size=(0.15, 0.15, 0.15),
    freejoint = True,
    p=(0.2, 0, 0.15),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 0.3, 0.5),
    group=1,
    friction=(1.0, 0.005, 0.0001),
    mass=0.5
)
spec_helper.add_site(
    name="contact_right",
    size=(0.05,0.05,0.05),
    p=(0, 0.15+0.05, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)
spec_helper.add_site(
    name="contact_left",
    size=(0.05,0.05,0.05),
    p=(0, -0.15-0.05, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)
spec_helper.add_site(
    name="center",
    size=(0.05,0.05,0.05),
    p=(0, 0, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)

model, data = spec_helper.compile()
spec_helper.save_to_xml("../asset/xml/scene_panda_lr.xml")

#### 1. Initialize Scene

In [3]:
joint_names = get_joint_names(model, data)
joint_names_left = [name for name in joint_names if name is not None and "_left" in name]
joint_names_right = [name for name in joint_names if name is not None and "_right" in name]
actuator_names = get_actuator_names(model, data)
actuator_names_left = [name for name in actuator_names if name is not None and "_left" in name]
actuator_names_right = [name for name in actuator_names if name is not None and "_right" in name]

""" GO TO INITIAL QPOS """
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

In [4]:
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # mujoco.mj_step(model, data)
    mujoco.mj_kinematics(model, data)
    # mujoco.mj_forward(model, data)
    viewer.render()

viewer.close()
del(viewer)

#### 2. Generate Trajectory with sequence of EE pose
- Define sequence of EE pose
- Solve sequence of IK

In [5]:
# end effector rotation target 
R_target_ee_left = [[1.0, 0.0, 0.0],[0.0, 0.0, 1.0],[0.0, -1.0, 0.0]]
R_target_ee_right = [[1.0, 0.0, 0.0],[0.0, 0.0, -1.0],[0.0, 1.0, 0.0]]

In [6]:
def solve_ik_bimanual(
        p_target_ee_left,
        p_target_ee_right,
        qpos_init_left,
        qpos_init_right
):
    mujoco.mj_resetData(model, data)
    apply_qpos_names(model, data, names=joint_names_left, value=qpos_init_left)
    apply_qpos_names(model, data, names=joint_names_right, value=qpos_init_right)
    mujoco.mj_forward(model, data)

    while True:
        # get current EE position & rotation
        p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
        p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
        R_ee_left = get_R(model, data, name="eef_sphere_left", type='geom')
        R_ee_right = get_R(model, data, name="eef_sphere_right", type='geom')
        # calculate ik error 
        pos_error_left, rotvec_error_left = get_ik_error_clipped(
            p_target=p_target_ee_left,
            r_target=R_target_ee_left,
            p_current=p_ee_left,
            r_current=R_ee_left,
            )
        pos_error_right, rotvec_error_right = get_ik_error_clipped(
            p_target=p_target_ee_right,
            r_target=R_target_ee_right,
            p_current=p_ee_right,
            r_current=R_ee_right,
            )
        error_left = np.concatenate([pos_error_left, rotvec_error_left])
        error_right = np.concatenate([pos_error_right, rotvec_error_right])

        # terminate condition
        # if np.linalg.norm(pos_error_left) < 0.001 and np.linalg.norm(rotvec_error_left) < 0.001 and np.linalg.norm(rotvec_error_right) < 0.001 and np.linalg.norm(pos_error_right) < 0.001:
        if np.linalg.norm(pos_error_left) < 0.001 and np.linalg.norm(pos_error_right) < 0.001:
            qpos_left = get_qpos_with_names(model, data, names=joint_names_left)
            qpos_right = get_qpos_with_names(model, data, names=joint_names_right)
            return qpos_left, qpos_right

        # calculate jacobian
        jac_p_left, jac_r_left = get_jacobian(model, data, 'eef_sphere_left', type='geom', joints_use=joint_names_left)
        jac_p_right, jac_r_right = get_jacobian(model, data, 'eef_sphere_right', type='geom', joints_use=joint_names_right)
        jac_left = np.concatenate([jac_p_left, jac_r_left], axis=0)
        jac_right = np.concatenate([jac_p_right, jac_r_right], axis=0)
        jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
        jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)
        # calculate qpos error
        qpos_error_left = jac_left_inverse @ error_left
        qpos_error_right = jac_right_inverse @ error_right
        q_left_updated = get_qpos_with_names(model, data, names=joint_names_left) + qpos_error_left
        q_right_updated = get_qpos_with_names(model, data, names=joint_names_right) + qpos_error_right
        # update qpos
        apply_qpos_names(model, data, names=joint_names_left, value=q_left_updated)
        apply_qpos_names(model, data, names=joint_names_right, value=q_right_updated)
        mujoco.mj_forward(model, data)

Trajectory - upward direction

In [7]:
# 0. list of qpos 
q_left_list = []
q_right_list = []
p_ee_left_1 = get_p(model, data, name="contact_left", type='site')
p_ee_right_1 = get_p(model, data, name="contact_right", type='site')

# 1. first pose 
q_left, q_right = solve_ik_bimanual(
    p_target_ee_left = p_ee_left_1,
    p_target_ee_right = p_ee_right_1,
    qpos_init_left = qpos_init,
    qpos_init_right = qpos_init
)
q_left_list.append(q_left)
q_right_list.append(q_right)

In [8]:
p_ee_left = p_ee_left_1
p_ee_right = p_ee_right_1

# 1. upward trajectory 
for i in range(200):
    p_ee_left = p_ee_left + np.array([0, 0, 0.001])
    p_ee_right = p_ee_right + np.array([0, 0, 0.001])
    q_left, q_right = solve_ik_bimanual(
        p_target_ee_left = p_ee_left,
        p_target_ee_right = p_ee_right,
        qpos_init_left = q_left,
        qpos_init_right = q_right
    )
    print(f"\r Step {i+1}/200 completed, p_ee_left: {p_ee_left}, p_ee_right: {p_ee_right}", end="")
    q_left_list.append(q_left)
    q_right_list.append(q_right)

# 2. backward trajectory 
for i in range(400):
    p_ee_left = p_ee_left + np.array([-0.001, 0, 0])
    p_ee_right = p_ee_right + np.array([-0.001, 0, 0])
    q_left, q_right = solve_ik_bimanual(
        p_target_ee_left = p_ee_left,
        p_target_ee_right = p_ee_right,
        qpos_init_left = q_left,
        qpos_init_right = q_right
    )
    print(f"\r Step {i+1}/400 completed, p_ee_left: {p_ee_left}, p_ee_right: {p_ee_right}", end="")
    q_left_list.append(q_left)
    q_right_list.append(q_right)

# 3. downward trajectory
for i in range(200):
    p_ee_left = p_ee_left + np.array([0, 0, -0.001])
    p_ee_right = p_ee_right + np.array([0, 0, -0.001])
    q_left, q_right = solve_ik_bimanual(
        p_target_ee_left = p_ee_left,
        p_target_ee_right = p_ee_right,
        qpos_init_left = q_left,
        qpos_init_right = q_right
    )
    print(f"\r Step {i+1}/200 completed, p_ee_left: {p_ee_left}, p_ee_right: {p_ee_right}", end="")
    q_left_list.append(q_left)
    q_right_list.append(q_right)

print("q list shape (left):", len(q_left_list))
print("q list shape (right):", len(q_right_list))

 Step 200/200 completed, p_ee_left: [-0.2  -0.2   0.15], p_ee_right: [-0.2   0.2   0.15]q list shape (left): 801 2.0000000e-01  3.5000000e-01]
q list shape (right): 801


In [9]:
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
viewer.options[0].flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=q_left_list[0])
apply_qpos_names(model, data, names=joint_names_right, value=q_right_list[0])
mujoco.mj_forward(model, data)

idx = 0
time_init = time.time()
while viewer.is_alive():
    if time.time() - time_init > 0.01:
        idx += 1
        time_init = time.time()
    if idx >= len(q_left_list):
        idx = len(q_left_list) - 1
    apply_qpos_names(model, data, names=joint_names_left, value=q_left_list[idx])
    apply_qpos_names(model, data, names=joint_names_right, value=q_right_list[idx])
    mujoco.mj_forward(model, data)
    print ("\r index:", idx, end="")
    viewer.render()

viewer.close()
del(viewer)

 index: 800

#### 3. PID Control - Follow Trajectory

In [10]:
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=q_left_list[0])
apply_qpos_names(model, data, names=joint_names_right, value=q_right_list[0])
mujoco.mj_forward(model, data)

In [22]:
q_current_left = get_qpos_with_names(model, data, names=joint_names_left)
q_current_right = get_qpos_with_names(model, data, names=joint_names_right)
q_target_left = q_left_list[1]
q_target_right = q_right_list[1]
q_diff_left = q_target_left - q_current_left
q_diff_right = q_target_right - q_current_right

# PID controller
Kp = 10.0
Ki = 1.0
Kd = 5.0
error_sum_left = np.zeros_like(q_diff_left)
error_sum_right = np.zeros_like(q_diff_right)
error_prev_left = np.zeros_like(q_diff_left)
error_prev_right = np.zeros_like(q_diff_right)

error_sum_left += q_diff_left
error_sum_right += q_diff_right
error_derivative_left = q_diff_left - error_prev_left
error_derivative_right = q_diff_right - error_prev_right

control_signal_left = Kp * q_diff_left + Ki * error_sum_left + Kd * error_derivative_left
control_signal_right = Kp * q_diff_right + Ki * error_sum_right + Kd * error_derivative_right

In [23]:
# loop 
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
viewer.options[0].flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=q_left_list[0])
apply_qpos_names(model, data, names=joint_names_right, value=q_right_list[0])
mujoco.mj_forward(model, data)

idx = 0
q_goal_left = q_left_list[idx]
q_goal_right = q_right_list[idx]
error_sum_left = np.zeros_like(q_diff_left)
error_sum_right = np.zeros_like(q_diff_right)
error_prev_left = np.zeros_like(q_diff_left)
error_prev_right = np.zeros_like(q_diff_right)

while viewer.is_alive():
    q_current_left = get_qpos_with_names(model, data, names=joint_names_left)
    q_current_right = get_qpos_with_names(model, data, names=joint_names_right)
    q_diff_left = q_goal_left - q_current_left
    q_diff_right = q_goal_right - q_current_right
    while (np.linalg.norm(q_diff_left) < 0.01 and np.linalg.norm(q_diff_right) < 0.01):
        idx += 1
        q_goal_left = q_left_list[idx]
        q_goal_right = q_right_list[idx]
        q_diff_left = q_goal_left - q_current_left
        q_diff_right = q_goal_right - q_current_right
    error_sum_left += q_diff_left * 0.01
    error_sum_right += q_diff_right * 0.01
    error_derivative_left = q_diff_left - error_prev_left
    error_derivative_right = q_diff_right - error_prev_right

    control_signal_left = Kp * q_diff_left + Ki * error_sum_left + Kd * error_derivative_left
    control_signal_right = Kp * q_diff_right + Ki * error_sum_right + Kd * error_derivative_right
    # bias force (use dof address index)
    joint_dofadr_left = [model.jnt_dofadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)] for name in joint_names_left]
    joint_dofadr_right = [model.jnt_dofadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)] for name in joint_names_right]
    q_frc_bias_left = data.qfrc_bias[joint_dofadr_left]
    q_frc_bias_right = data.qfrc_bias[joint_dofadr_right]
    total_torque_left = q_frc_bias_left + control_signal_left * 0.001
    total_torque_right = q_frc_bias_right + control_signal_right * 0.001

    apply_ctrl_names(model, data, names=actuator_names_left, value=total_torque_left)
    apply_ctrl_names(model, data, names=actuator_names_right, value=total_torque_right)
    error_prev_left = q_diff_left
    error_prev_right = q_diff_right

    mujoco.mj_step(model, data)
    viewer.render()

viewer.close()
del(viewer)

In [ ]:
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

while True:
    # get current EE position & rotation
    p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
    p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
    R_ee_left = get_R(model, data, name="eef_sphere_left", type='geom')
    R_ee_right = get_R(model, data, name="eef_sphere_right", type='geom')
    # get target p (R is globally fixed )
    p_target_ee_left = get_p(model, data, name="contact_left", type='site')
    p_target_ee_right = get_p(model, data, name="contact_right", type='site')
    # calculate ik error 
    pos_error_left, rotvec_error_left = get_ik_error_clipped(
        p_target=p_target_ee_left,
        r_target=R_target_ee_left,
        p_current=p_ee_left,
        r_current=R_ee_left,
        )
    pos_error_right, rotvec_error_right = get_ik_error_clipped(
        p_target=p_target_ee_right,
        r_target=R_target_ee_right,
        p_current=p_ee_right,
        r_current=R_ee_right,
        )
    error_left = np.concatenate([pos_error_left, rotvec_error_left])
    error_right = np.concatenate([pos_error_right, rotvec_error_right])

    # terminate condition
    if np.linalg.norm(pos_error_left) < 0.01 and np.linalg.norm(rotvec_error_left) < 0.01 and np.linalg.norm(rotvec_error_right) < 0.01 and np.linalg.norm(pos_error_right) < 0.01:
        qpos_left = get_qpos_with_names(model, data, names=joint_names_left)
        qpos_right = get_qpos_with_names(model, data, names=joint_names_right)
        print("\r right arm Target reached!", end="")
        break 

    # calculate jacobian
    jac_p_left,jac_r_left = get_jacobian(model, data, 'eef_sphere_right', type='geom', joints_use=joint_names_right)
    jac_p_right, jac_r_right = get_jacobian(model, data, 'eef_sphere_left', type='geom', joints_use=joint_names_left)
    jac_left = np.concatenate([jac_p_left, jac_r_left], axis=0)
    jac_right = np.concatenate([jac_p_right, jac_r_right], axis=0)
    jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
    jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)
    # calculate qpos error
    qpos_error_left = jac_left_inverse @ error_left
    qpos_error_right = jac_right_inverse @ error_right
    print("qpos error (left):", qpos_error_left)
    print("qpos error (right):", qpos_error_right)
    q_left_updated = get_qpos_with_names(model, data, names=joint_names_left) + qpos_error_left
    q_right_updated = get_qpos_with_names(model, data, names=joint_names_right) + qpos_error_right
    # update qpos
    apply_qpos_names(model, data, names=joint_names_left, value=q_left_updated)
    apply_qpos_names(model, data, names=joint_names_right, value=q_right_updated)
    mujoco.mj_forward(model, data)

qpos error (left): [ 0.04685293 -0.01484621 -0.02665555 -0.0345848   0.20589594 -0.10816937
  0.0656855 ]
qpos error (right): [-0.09430104 -0.08922464  0.06932736 -0.15232313 -0.1833973   0.19100645
  0.17076761]
qpos error (left): [ 0.07074471 -0.04104037 -0.03358059 -0.06662905  0.20050375 -0.02381594
  0.06122833]
qpos error (right): [-0.08333402 -0.0716043   0.065751   -0.1122838  -0.15857971  0.09812234
  0.15438728]
qpos error (left): [ 0.08838135 -0.05255648 -0.04516485 -0.08457819  0.19026035  0.02639275
  0.06035204]
qpos error (right): [-0.08421356 -0.06276445  0.06565811 -0.08979101 -0.15259607  0.04929066
  0.1535771 ]
qpos error (left): [ 0.09848604 -0.06183073 -0.05799235 -0.09917896  0.1741764   0.07081179
  0.06331607]
qpos error (right): [-0.08930632 -0.05679271  0.06484641 -0.07438041 -0.15790052  0.0124117
  0.16135632]
qpos error (left): [ 0.10043459 -0.07547598 -0.07403518 -0.11482044  0.14827612  0.12453647
  0.07171561]
qpos error (right): [-0.09911264 -0.0503367

#### 4-1. End effector position: Cartesian Impedance controller
- Declare gains & jacobians
- position targets & differences

In [ ]:
# Gains
Kp_ee = 100.0
Kd_ee = 20.0
Kp_force = 10.0
Ki_force = 1.0
# get jacobian transpose
jac_p_left,jac_r_left = get_jacobian(model, data, 'eef_sphere_right', type='geom', joints_use=joint_names_right)
jac_p_right, jac_r_right = get_jacobian(model, data, 'eef_sphere_left', type='geom', joints_use=joint_names_left)
jac_left = np.concatenate([jac_p_left, jac_r_left], axis=0)
jac_right = np.concatenate([jac_p_right, jac_r_right], axis=0)
# pseudo inverse of positional jacobian
jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)

In [ ]:
# get positional error
p_ee_target_left = get_p(model, data, name="contact_left", type='site')
p_ee_target_right = get_p(model, data, name="contact_right", type='site')
p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
p_ee_left_error = p_ee_target_left - p_ee_left
p_ee_right_error = p_ee_target_right - p_ee_right

v_ee_target_left = np.zeros(3)
v_ee_target_right = np.zeros(3)
qvel_left = get_qvel_with_names(model, data, names=joint_names_left)
qvel_right = get_qvel_with_names(model, data, names=joint_names_right)
v_ee_left = jac_p_left @ qvel_left
v_ee_right = jac_p_right @ qvel_right
v_ee_left_error = v_ee_target_left - v_ee_left
v_ee_right_error = v_ee_target_right - v_ee_right

# calculate torque 
f_ee_desired_left = Kp_ee * p_ee_left_error + Kd_ee * v_ee_left_error
f_ee_desired_right = Kp_ee * p_ee_right_error + Kd_ee * v_ee_right_error
torque_left = jac_p_left.T @ f_ee_desired_left
torque_right = jac_p_right.T @ f_ee_desired_right

#### 4-2. Push force: PI controller for cartesian force
- Force direction target with site positions
- Force magnitude: calculate with friction

In [ ]:
def get_body_contact_force_position(
        model,
        data,
        body1_name,
        body2_name,
        ):
    body1_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body1_name)
    body2_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body2_name)
    contact_forces = []
    contact_positions = []
    for i in range(data.ncon):
        contact = data.contact[i]
        body1_in_contact_id = model.geom_bodyid[contact.geom1]
        body2_in_contact_id = model.geom_bodyid[contact.geom2]
        if (body1_in_contact_id == body1_id and body2_in_contact_id == body2_id) or (body1_in_contact_id == body2_id and body2_in_contact_id == body1_id):
            force = np.zeros(6) # (6,)
            mujoco.mj_contactForce(model,data,i,force)
            contact_forces.append(force)
            contact_positions.append(contact.pos)
    return contact_forces, contact_positions

In [ ]:
# get desired cartesian force direction & magnitude 
p_center = get_p(model, data, name="center", type='site')
direction_push_left = (p_center - p_target_contact_left) / np.linalg.norm(p_center - p_target_contact_left)
direction_push_right = (p_center - p_target_contact_right) / np.linalg.norm(p_center - p_target_contact_right)
force_magnitude = 5.0 # 5 newtons 
f_push_target_left = Kp_force * (force_magnitude * direction_push_left)
f_push_target_right = Kp_force * (force_magnitude * direction_push_right)
# get current contact force
f_push_left, p_contact_left = get_body_contact_force_position(model, data, body1_name="eef_sphere_left", body2_name="box")
f_push_right, p_contact_right = get_body_contact_force_position(model, data, body1_name="eef_sphere_right", body2_name="box")
print("Current contact force (left):", f_push_left, "Contact position (left):", p_contact_left)
print("Current contact force (right):", f_push_right, "Contact position (right):", p_contact_right)

# process contact 
if len(f_push_left) == 0:
    f_push_left = np.zeros(3)
else:
    f_push_left = f_push_left[0][:3] # take the first contact and only force part
if len(f_push_right) == 0:
    f_push_right = np.zeros(3)
else:
    f_push_right = f_push_right[0][:3] # take the first contact and only force part

# desired force with PI controller
f_accumulated_left = np.zeros(3)
f_accumulated_right = np.zeros(3)
f_push_error_left = f_push_target_left - f_push_left
f_push_error_right = f_push_target_right - f_push_right
f_accumulated_left += f_push_error_left * 0.01 # integral term with dt=0.01s
f_accumulated_right += f_push_error_right * 0.01

# get torque
f_push_desired_left = f_push_target_left + Kp_force * f_push_error_left + Ki_force * f_accumulated_left
f_push_desired_right = f_push_target_right + Kp_force * f_push_error_right + Ki_force * f_accumulated_right
torque_left = jac_p_left.T @ f_push_desired_left
torque_right = jac_p_right.T @ f_push_desired_right

NameError: name 'p_target_contact_left' is not defined

#### 5. Iterate
- Iteratively apply two torques

In [ ]:
actuator_names = get_actuator_names(model, data)
print("Actuator names:", actuator_names)
actuator_names_left = [name for name in actuator_names if name is not None and "_left" in name]
print("Left actuator names:", actuator_names_left)
actuator_names_right = [name for name in actuator_names if name is not None and "_right" in name]
print("Right actuator names:", actuator_names_right)

In [ ]:
print("bias force:", data.qfrc_bias)

In [ ]:
Kp_ee = 100.0
Kd_ee = 20.0
Kp_force = 10.0
Ki_force = 1.0
force_magnitude = 5.0 # 5 newtons 

viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
viewer.options[0].flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_left_saved)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_right_saved)
mujoco.mj_forward(model, data)

while viewer.is_alive():
    # get jacobian transpose
    jac_p_left,jac_r_left = get_jacobian(model, data, 'eef_sphere_right', type='geom', joints_use=joint_names_right)
    jac_p_right, jac_r_right = get_jacobian(model, data, 'eef_sphere_left', type='geom', joints_use=joint_names_left)
    jac_left = np.concatenate([jac_p_left, jac_r_left], axis=0)
    jac_right = np.concatenate([jac_p_right, jac_r_right], axis=0)
    jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
    jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)
    
    # pd control torque 
    p_ee_target_left = get_p(model, data, name="contact_left", type='site')
    p_ee_target_right = get_p(model, data, name="contact_right", type='site')
    p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
    p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
    p_ee_left_error = p_ee_target_left - p_ee_left
    p_ee_right_error = p_ee_target_right - p_ee_right
    v_ee_target_left = np.zeros(3)
    v_ee_target_right = np.zeros(3)
    qvel_left = get_qvel_with_names(model, data, names=joint_names_left)
    qvel_right = get_qvel_with_names(model, data, names=joint_names_right)
    v_ee_left = jac_p_left @ qvel_left
    v_ee_right = jac_p_right @ qvel_right
    v_ee_left_error = v_ee_target_left - v_ee_left
    v_ee_right_error = v_ee_target_right - v_ee_right
    f_ee_desired_left = Kp_ee * p_ee_left_error + Kd_ee * v_ee_left_error
    f_ee_desired_right = Kp_ee * p_ee_right_error + Kd_ee * v_ee_right_error
    torque_ee_left = jac_p_left.T @ f_ee_desired_left
    torque_ee_right = jac_p_right.T @ f_ee_desired_right

    # get push force torque 
    p_center = get_p(model, data, name="center", type='site')
    direction_push_left = (p_center - p_target_contact_left) / np.linalg.norm(p_center - p_target_contact_left)
    direction_push_right = (p_center - p_target_contact_right) / np.linalg.norm(p_center - p_target_contact_right)
    f_push_target_left = Kp_force * (force_magnitude * direction_push_left)
    f_push_target_right = Kp_force * (force_magnitude * direction_push_right)
    f_push_left, p_contact_left = get_body_contact_force_position(model, data, body1_name="eef_sphere_left", body2_name="box")
    f_push_right, p_contact_right = get_body_contact_force_position(model, data, body1_name="eef_sphere_right", body2_name="box")
    if len(f_push_left) == 0:
        f_push_left = np.zeros(3)
    else:
        f_push_left = f_push_left[0][:3] # take the first contact and only force part
    if len(f_push_right) == 0:
        f_push_right = np.zeros(3)
    else:
        f_push_right = f_push_right[0][:3] # take the first contact and only force part
    f_accumulated_left = np.zeros(3)
    f_accumulated_right = np.zeros(3)
    f_push_error_left = f_push_target_left - f_push_left
    f_push_error_right = f_push_target_right - f_push_right
    f_accumulated_left += f_push_error_left * 0.01 # integral term with dt=0.01s
    f_accumulated_right += f_push_error_right * 0.01
    f_push_desired_left = f_push_target_left + Kp_force * f_push_error_left + Ki_force * f_accumulated_left
    f_push_desired_right = f_push_target_right + Kp_force * f_push_error_right + Ki_force * f_accumulated_right
    torque_push_left = jac_p_left.T @ f_push_desired_left
    torque_push_right = jac_p_right.T @ f_push_desired_right

    # bias force (use dof address index)
    joint_dofadr_left = [model.jnt_dofadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)] for name in joint_names_left]
    joint_dofadr_right = [model.jnt_dofadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)] for name in joint_names_right]
    q_frc_bias_left = data.qfrc_bias[joint_dofadr_left]
    q_frc_bias_right = data.qfrc_bias[joint_dofadr_right]

    # total torque
    calculated_torque_left = (torque_ee_left + torque_push_left)*0.1 # *1e-1
    calculated_torque_right = (torque_ee_right + torque_push_right)*0.1
    total_torque_left = calculated_torque_left + q_frc_bias_left
    total_torque_right = calculated_torque_right + q_frc_bias_right
    # total_torque_left = q_frc_bias_left
    # total_torque_right = q_frc_bias_right
    apply_ctrl_names(model, data, names=actuator_names_left, value = total_torque_left)
    apply_ctrl_names(model, data, names=actuator_names_right, value = total_torque_right)

    mujoco.mj_step(model, data)
    viewer.render()
    # time.sleep(0.1)

viewer.close()
del(viewer)